<a href="https://colab.research.google.com/github/TuulaP/imgcat/blob/main/natlibfi_imgcateg_tf2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Experimental historical illustration basic categorization work

Experimental historical newspaper illustration basic categorization demo.

Acknowledgments

<table><tr><td>
<img src="https://digi.kansalliskirjasto.fi/images/sosiaali_en.png" alt="European Regional Development Fund" height="110">
</td><td>
<img src="https://digi.kansalliskirjasto.fi/images/en_EU_rgb.png" alt="Leverage from the EU 2014-2020" height="110"></td>
</tr></table>

#### Setup

**Note:** This notebook has been updated to use TensorFlow 2.x (using  for frozen  graph loading).

Download the image classifier and the labels.

Note! the download links below will inactivate at December 2019, after which please download the packages (Illustration base type classifier model file & labels) from https://digi.kansalliskirjasto.fi/opendata and update links below.

In [16]:


!curl https://digi.kansalliskirjasto.fi/opendata-download-file/13534b04a96b4f03b405be21698c51c3/nlf_basetype_classifier.pb --output nlf_basetype_classifier.pb

!curl https://digi.kansalliskirjasto.fi/opendata-download-file/4b9fde78245d4cfc99c1ffe607f3fa73/nlf_basetype_classifier_labels.txt  --output nlf_basetype_classifier_labels.txt



  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 83.4M  100 83.4M    0     0  3503k      0  0:00:24  0:00:24 --:--:-- 5833k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    77  100    77    0     0    111      0 --:--:-- --:--:-- --:--:--   111


###  Image categorization code

Basic imports

In [17]:
# TF2: removed %tensorflow_version 1.x magic (Colab-only, TF1 specific)
import tensorflow as tf
import sys
import glob
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Disable TF2 eager execution to allow loading a frozen .pb graph
tf.compat.v1.disable_eager_execution()

# setup for label changes
usedstrings = {
    'piirros painokuva': 'piirrospainokuva',
    'teksti mainos': 'mainos',
    'roskat': 'muut'
}


In [18]:
# The function to do the categorization, the img parameter carries the filename

def luokittelekuva(img=sys.argv[1]):
    # TF2: use tf.io.gfile instead of tf.gfile
    image_data = tf.io.gfile.GFile(img, 'rb').read()

    AIDIR = ""  # for codecollab
    image_path = "./tmp/"

    # Read labels
    # TF2: use tf.io.gfile instead of tf.gfile
    label_lines = [line.rstrip() for line
                   in tf.io.gfile.GFile(AIDIR + "nlf_basetype_classifier_labels.txt")]

    # Load frozen graph
    # TF2: use tf.compat.v1.GraphDef and a new explicit Graph object
    graph_def = tf.compat.v1.GraphDef()
    with tf.io.gfile.GFile(AIDIR + "nlf_basetype_classifier.pb", 'rb') as f:
        graph_def.ParseFromString(f.read())

    graph = tf.compat.v1.Graph()
    with graph.as_default():
        tf.compat.v1.import_graph_def(graph_def, name='')

    # Run inference
    # TF2: use tf.compat.v1.Session with the explicit graph
    with tf.compat.v1.Session(graph=graph) as sess:
        softmax_tensor = sess.graph.get_tensor_by_name('final_result:0')
        predictions = sess.run(softmax_tensor,
                               {'DecodeJpeg/contents:0': image_data})

        # Sort to show labels of first prediction in order of confidence
        limit_score = 0.6
        top_k = predictions[0].argsort()[-len(predictions[0]):][::-1]

        max_score = 0
        max_id = 0
        for node_id in top_k:
            score = predictions[0][node_id]
            if max_score < score:
                max_score = score
                max_id = node_id

        human_string = label_lines[max_id]
        print('%s : %s (score = %.5f)' % (image_path, human_string, score))

        if human_string in usedstrings:
            human_string = usedstrings[human_string]

        return (human_string, score)


One expected drawing type:

In [19]:

!curl https://digi.kansalliskirjasto.fi/sanomalehti/binding/1198629/articles/43569800/images/1437505?scale=1.0 --output kuva1.jpg


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 40245    0 40245    0     0  38497      0 --:--:--  0:00:01 --:--:-- 38511


In [20]:
# The actual way to run the function

luo, score = luokittelekuva("kuva1.jpg") # sys.argv[1])


./tmp/ : piirros painokuva (score = 0.00913)


In [21]:
print("Category: {0}, score {1} ".format(luo, score))

Category: piirrospainokuva, score 0.00912800058722496 


Illustration of unknown type:

In [22]:
!curl https://digi.kansalliskirjasto.fi/aikakausi/binding/1355524/articles/78396775/images/120919660?scale=1.0 --output kuva2.jpg

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 87108    0 87108    0     0  55387      0 --:--:--  0:00:01 --:--:-- 55376


In [23]:
luo, score = luokittelekuva("kuva2.jpg") # sys.argv[1])
print("Category: {0}, score {1} ".format(luo, score))


./tmp/ : piirros painokuva (score = 0.00012)
Category: piirrospainokuva, score 0.00012121235340600833 


In [24]:

!curl https://digi.kansalliskirjasto.fi/sanomalehti/binding/679866/articles/1544609/images/10576494?scale=1.0 --output kuva3.jpg

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  838k    0  838k    0     0   217k      0 --:--:--  0:00:03 --:--:--  217k


In [25]:
# then not so succesfull classification...

luo, score = luokittelekuva("kuva3.jpg")
print("Category: {0}, score {1} ".format(luo, score))


./tmp/ : kartat (score = 0.01916)
Category: kartat, score 0.019164422526955605 
